# Register an MdGdalSource dataset by hand

`MdGdalSource` reads a GDAL **multidimensional** array as a raster. A netCDF variable with
dimensions `(time, lat, lon)` has no raster bands at all — `time` is just a dimension — so
registering one means stating how that dimension maps onto Geo Engine's two axes: the **z
axis** (`zRole`) and the **band** axis.

This notebook registers one small 3D fixture with **every metadata field written out by
hand**. Nothing is derived from the file, so what you see is exactly what the server stores
and reads. `python/examples/md_example.nc` is a byte-identical copy of
`geoengine/test_data/md/time_series.nc`, whose values follow the deterministic formula
`value = t*1000 + y*10 + x` — the verification at the end asserts exactly that.

In [ ]:
import asyncio
from pathlib import Path
from pathlib import Path

import geoengine as ge
import geoengine_api_client as geoc

ge.initialize("http://localhost:3030/api", ("admin@localhost", "adminadmin"))

# The fixture lives next to this notebook, so the example needs no configured volume.
FIXTURE = Path("data/md_example.nc").resolve()
assert FIXTURE.is_file(), FIXTURE
print(FIXTURE)

## Upload the file

`data_store` accepts a volume, an upload id or `"external"`. Uploading keeps this example
self-contained; a volume would work identically with a path relative to its root.

In [ ]:
session = ge.get_session()
with geoc.ApiClient(session.configuration) as api_client:
    uploads_api = geoc.UploadsApi(api_client)
    response = uploads_api.upload_handler([str(FIXTURE)], _request_timeout=600)

data_store = ge.UploadId.from_response(response)
print("uploaded:", data_store)

## Describe the array

Everything the server needs about the **dataset** as a whole. These are the fields of
`GdalMdMetaData`:

| field | meaning here |
|---|---|
| `resultDescriptor` | data type, CRS, time axis, grid, bands |
| `zRole` | `Variable` = the z slices are the **time axis**; `Band` = they are the **bands** |
| `wrap` | stored longitudes are `0..360` and are presented as `-180..180` |
| `maxZBatchSize` | upper bound on z slices one worker read may request |
| `cacheTtl` | dataset-level cache lifetime in seconds |

The grid deserves a note, because there are **two** transforms:

* `params.geoTransform` on each tile below is the grid **as stored in the file** — here
  descending `lat`, so a negative y pixel size.
* `resultDescriptor.spatialGrid` is the grid **as presented**, always north-up. The server
  flips the y axis when reading an ascending-latitude array, which is why the stored transform
  is the one that goes on the tile.

For this fixture they coincide (the file is already north-up), but for an ascending-latitude
file they must not, and the server rejects a tile whose presented transform does not land on
the dataset's.

In [ ]:
# The stored grid: pixel edges at lon 0..240 (30 degree pixels) and lat 0..-8, which is the
# 8x8 fixture's footprint. Cell *centres* are at 15, 45, ... 225 and -0.5, -1.5, ... -7.5,
# because netCDF/CF stores centres.
STORED_GEO_TRANSFORM = geoc.GeoTransform(
    origin_coordinate=geoc.Coordinate2D(x=0.0, y=0.0),
    x_pixel_size=30.0,
    y_pixel_size=-1.0,
)
WIDTH, HEIGHT = 8, 8

# The presented grid. North-up, so also -1.0 here.
result_descriptor = geoc.RasterResultDescriptor(
    data_type="F32",
    spatial_reference="EPSG:4326",
    time=geoc.TimeDescriptor(
        bounds=geoc.TimeInterval(start=946684800000, end=946684800000 + 8 * 86_400_000),
        dimension=geoc.TimeDimension(
            actual_instance=geoc.Regular(
                origin=946684800000,
                step=geoc.TimeStep(granularity="days", step=1),
                type="regular",
            )
        ),
    ),
    spatial_grid=geoc.SpatialGridDescriptor(
        spatialGrid=geoc.SpatialGridDefinition(
            geoTransform=STORED_GEO_TRANSFORM,
            gridBounds=geoc.GridBoundingBox2D(
                topLeftIdx=geoc.GridIdx2D(xIdx=0, yIdx=0),
                bottomRightIdx=geoc.GridIdx2D(xIdx=WIDTH - 1, yIdx=HEIGHT - 1),
            ),
        ),
        descriptor="source",
    ),
    measurement=geoc.Measurement(actual_instance=geoc.UnitlessMeasurement(type="unitless")),
    bands=[
        geoc.RasterBandDescriptor(
            name="temperature",
            measurement=geoc.Measurement(actual_instance=geoc.UnitlessMeasurement(type="unitless")),
        )
    ],
)

md = geoc.GdalMdMetaData(
    type="GdalMdMetaData",
    result_descriptor=result_descriptor,
    z_role="variable",
    wrap=False,
    maxZBatchSize=4,
    cacheTtl=None,
)

# `MetaDataDefinition` is a oneOf wrapper; the concrete metadata goes under `actual_instance`.
meta_data = geoc.MetaDataDefinition(actual_instance=md)
print(f"z_role={md.z_role}, wrap={md.wrap}, bands={[b.name for b in result_descriptor.bands]}")

## Describe the file

One row per MD array file, covering all of that file's z slices — `AddDatasetMdTile`. A 3D
array needs `leadingPrefix = []`; a 4D `(time, depth, lat, lon)` array would use `[2]` to
read depth 2, and one dataset can hold a different depth per band.

Two fields are worth calling out:

* `timeSteps` is **one interval per z slice** — 8 here, one per day. It is what the read path
  derives the file's slice count from, so it cannot be a single interval for the whole file.
* `timeDescriptor` is the compact form the API advertises, and it must describe `timeSteps`.
  The server rejects a row where the two disagree, on every data path.

`spatialPartition` is the row's presented footprint; the server recomputes the dataset's own
extents from these rows.

In [ ]:
DAY_MS = 86_400_000
FIRST_DAY_MS = 946684800000  # 2000-01-01T00:00:00Z
time_steps = [
    geoc.TimeInterval(start=FIRST_DAY_MS + i * DAY_MS, end=FIRST_DAY_MS + (i + 1) * DAY_MS)
    for i in range(8)
]

# the row's presented footprint: pixel edges at lon 0..240, lat -8..0
# The row's presented footprint, in *spatial* coordinates (the server converts the tile's
# grid indices itself). Pixel edges span lon 0..240 and lat -8..0.
spatial_partition = geoc.SpatialPartition2D(
    upperLeftCoordinate=geoc.Coordinate2D(x=0.0, y=0.0),
    lowerRightCoordinate=geoc.Coordinate2D(x=240.0, y=-8.0),
)

tiles = [
    geoc.AddDatasetMdTile(
        spatial_partition=spatial_partition,
        band=0,
        z_index=0,
        array_name="temperature",
        arrayGroup=None,
        time_descriptor=geoc.TimeDescriptor(
            bounds=geoc.TimeInterval(start=time_steps[0].start, end=time_steps[-1].end),
            dimension=geoc.TimeDimension(
                actual_instance=geoc.Regular(
                    origin=time_steps[0].start,
                    step=geoc.TimeStep(granularity="days", step=1),
                    type="regular",
                )
            ),
        ),
        time_steps=time_steps,
        params=geoc.GdalDatasetParameters(
            file_path="md_example.nc",  # relative to the upload's root
            rasterband_channel=1,
            geo_transform=STORED_GEO_TRANSFORM,
            width=WIDTH,
            height=HEIGHT,
            file_not_found_handling="NoData",
            noDataValue=-9999.0,
        ),
        leadingPrefix=[],
    )
]
print(f"{len(tiles)} row, {len(tiles[0].time_steps)} time steps")

## Register

Two calls: the dataset with its metadata, then its rows.

In [ ]:
dataset = ge.add_or_replace_dataset_with_permissions(
    data_store,
    ge.AddDatasetProperties(
        name="md_example",
        display_name="MD example (hand-written metadata)",
        description="An 8x8 daily netCDF array registered as an MdGdalSource, without any probing.",
        source_operator="MdGdalSource",
        symbology=ge.RasterSymbology(
            raster_colorizer=ge.SingleBandRasterColorizer(
                band=0,
                band_colorizer=ge.Colorizer.linear_with_mpl_cmap(color_map="viridis", min_max=(0.0, 8000.0), n_steps=10),
            ),
        ),
    ),
    meta_data,
    permission_tuples=[
        (ge.permissions.ANONYMOUS_USER_ROLE_ID, ge.Permission.READ),
        (ge.permissions.REGISTERED_USER_ROLE_ID, ge.Permission.READ),
    ],
    replace_existing=True,
    timeout=600,
)

ge.add_md_dataset_tiles(dataset, tiles, timeout=600)
print("registered:", dataset)

## Read the pixels back

The check that matters: the query window is in **geographic** coordinates while the fixture's
formula is in **stored index** space, so a wrong `geoTransform` would land the window
somewhere else and the values would not match. That makes this a real check on the
hand-written metadata rather than a round-trip tautology.

In [ ]:
import datetime as dt
import numpy as np

# typed operators, not a raw dict - the same path a real workflow uses
workflow = ge.register_workflow(workflow=ge.workflow_builder.operators.MdGdalSource(dataset=dataset))

# An *absolute* window, not one derived from the dataset's own bounds: that is what catches a
# wrong time origin, since nothing on the server compares the declared axis with the file's.
window = ge.TimeInterval(
    start=dt.datetime(2000, 1, 3, tzinfo=dt.timezone.utc),
    end=dt.datetime(2000, 1, 6, tzinfo=dt.timezone.utc),
)
bbox = ge.QueryRectangle(ge.BoundingBox2D(0.0, -8.0, 240.0, 0.0), window)

tiles_read = [tile async for tile in workflow.raster_stream(bbox, open_timeout=600)]

# the dataset is 8x8, so it fits inside one 512x512 tiling cell and each returned tile holds
# one day; the rest of that cell is the tiling grid's padding
print(f"{len(tiles_read)} tiles")
assert len(tiles_read) == 3, len(tiles_read)
for t in sorted(tiles_read, key=lambda tile: tile.time_start_ms):
    day = (np.datetime64(t.time_start_ms, 'ms').astype('datetime64[D]') - np.datetime64('2000-01-01', 'D')).astype(int)
    assert np.timedelta64(t.time_end_ms - t.time_start_ms, 'ms') == np.timedelta64(DAY_MS, 'ms'), t.time
    # the dataset's pixels are the tile's upper-left 8x8 block
    block = t.to_numpy_data_array()[:HEIGHT, :WIDTH]
    # the query is in geographic coordinates while the fixture's formula is in stored index
    # space, so a wrong geoTransform would read different cells and fail this assert
    expected = np.fromfunction(lambda y, x: day * 1000 + y * 10 + x, (HEIGHT, WIDTH), dtype=int)
    np.testing.assert_allclose(block, expected)
    print(f"  day {day}: [{block.min():.0f} .. {block.max():.0f}]  matches t*1000 + y*10 + x")

## Clean up

In [ ]:
ge.datasets.delete_dataset(dataset)
ge.reset()
print("deleted:", dataset)